# <a id='toc1_'></a>[Building the index](#toc0_)

Authors:
- Ramon Casas i Luque
- Marina Velasco Barriuso
- Shaoya Zhang

>[!NOTE] Run locally so ../../data/ resolves correctly from the project root.

**Table of contents**<a id='toc0_'></a>    
- [Part 0. Import libraries and data](#toc2_)    
- [Part 1: Indexing](#toc3_)    
  - [1. Index creation](#toc3_1_)    
  - [2. Test queries](#toc3_2_)    
  - [3. TF-IDF algorithm](#toc3_3_)    
- [Part 2](#toc4_)    
  - [1. Implementation of evaluation metrics](#toc4_1_)    
  - [2. Applying the evaluation metrics](#toc4_2_)    
  - [3. Ground truth for our test queries](#toc4_3_)    
    - [a. Labelling results manually](#toc4_3_1_)    
    - [b. Evaluation metrics for our test queries](#toc4_3_2_)    
    - [c. Analysis of the current system](#toc4_3_3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc2_'></a>[Part 0. Import libraries and data](#toc0_)

In [1]:
import os
import pandas as pd
import numpy as np
from numpy import linalg as la
import math
from collections import defaultdict, Counter
from array import array
import ast
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/mvelasco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# Open csv with preprocessed data
filename = "../../data/index_info.csv"
index_info = pd.read_csv(filename)

# List of fields, excluding 'pid': 'title', 'description', etc.
fields = index_info.columns.to_list()
fields.remove('pid')

# Parse fields as lists of terms instead of plain strings
for f in fields:
    index_info[f] = index_info[f].apply(ast.literal_eval)

# Sort by PID
index_info = index_info.sort_values('pid').reset_index(drop=True)

index_info.head(3)

,pid,title,description,brand,category,sub_category,seller,product_details
0,4hWtdShnw%3D%3D,"[men, regular, fit, solid, casual, shirt]",[],"[crock, cl]","[cloth, accessori]",[topwear],[crockscl],"[1, c328, 2, 38, regular, pure, cotton, full, ..."
1,ARME8P6GFSXGE9AW,"[welwear, suncool1008, nylon, arm, warmer, mul...","[tatto, sleev, protect, uv, raditaion, also, p...",[],"[cloth, accessori]","[winter, wear]","[navkar, craft]","[2, 2, arm, warmer, welwear, suncool1008, brow..."
2,ARME8P6GJQ8PZBE6,"[welwear, suncool1004, nylon, arm, warmer, mul...","[tatto, sleev, protect, uv, raditaion, also, p...",[],"[cloth, accessori]","[winter, wear]","[navkar, craft]","[1, 1, arm, warmer, welwear, suncool1004, blue..."


# <a id='toc3_'></a>[Part 1: Indexing](#toc0_)

## <a id='toc3_1_'></a>[1. Index creation](#toc0_)

In [3]:
def create_index(index_df: pd.DataFrame) -> defaultdict:  
    """
    Implement the inverted index
    
    :param index_df: DataFrame storing pre-processed product information 
    :returns index: the inverted index (implemented through a nested Python dictionary) containing terms and fields as keys and the corresponding
    list of documents where these keys appears in (and the positions) as values.
    """
    index = defaultdict(dict)

    # Iterate over each row (products)
    for i, row in index_df.iterrows():
        pid = row['pid'] # Product's ID

        # Build index only for this product and field
        # E.g. ['solid', 'women'] --> 'solid': {'title': [pid, [0]]}, 'women': {'title': [pid, [1]]}
        current_product_index = {}

        # Iterate over fields in this row: title, description, brand, ...
        for field in fields:

            terms = row[field] # List of preprocessed terms

            for position, term in enumerate(terms):
                if term in current_product_index.keys():

                    if field in current_product_index[term].keys():
                        # if the term is already in the current_product_index for the current page (current_page_current_product_index)
                        # append the position to the corresponding list
                        current_product_index[term][field][1].append(position)
                    else:
                        current_product_index[term][field] = [pid, array('I',[position])]              
                
                else:
                    # term is not in the dictionary yet
                    current_product_index[term] = {field: [pid, array('I',[position])]} #'I' indicates unsigned int (int in Python)}
                        
        # Merge the current product index with the main index
        for term, field_dict in current_product_index.items():
            for field, posting_page in field_dict.items():
                index[term].setdefault(field, []).append(posting_page)

    return index

index = create_index(index_info)

## <a id='toc3_2_'></a>[2. Test queries](#toc0_)

Our extremely creative (😉) test queries will be:
1. `men's cotton checkered shirt`
2. `multicolor women casual t-shirt`
3. `black beat sports shoes`
4. `animal print jacket for women`
5. `grey hoodie with pockets`

All queries include one commmon term such as `shirt`, `women`, `black`, or `jacket` plus some rarer terms that make the search more specific, like `checkered`, `casual`, or `animal print`.

In [4]:
# Preprocess the text of the description column
def build_terms(text):
    """
    Preprocess text removing stop words, stemming,
    transforming in lowercase and return the tokens of the text.

    :param text: to be preprocessed
    :returns: a list of tokens corresponding to the input text after the preprocessing
    """

    stemmer = PorterStemmer()
    tokenizer = RegexpTokenizer(r'\w+')             # keep only words & numbers, drop punctuation
    stop_words = set(stopwords.words("english"))
    for w in ['not', 'no']:
        stop_words.remove(w)                        # keep 'no' and 'not' to preserve meaning of sentences like "shirt with no sleeves"
    text= text.lower()                              # transform in lowercase
    text=  tokenizer.tokenize(text)                 # tokenize the text to get a list of terms
    text=[t for t in text if t not in stop_words]   # eliminate the stopwords
    text=[stemmer.stem(t) for t in text]            # perform stemming
    return text

In [5]:
def search(query, index):
    """
    The output is the list of documents that contain ALL of the query terms.
    So, we will get the list of documents for each query term, and take the intersection of them.

    :param query: Search query
    :param index: Index to use for the search
    :returns docs: List of document IDs that contain ALL of the query terms. 
    """
    query = build_terms(query)

    # Empty string case
    if not query:
        return []
    
    for i, term in enumerate(query):

        # term is not in index, there is no product with this query term
        if term not in index.keys():
            return []

        # store in term_pids the ids of the products that contain "term" in any field
        term_pids = set()
        for field in index[term].keys():
            for posting in index[term][field]:
                pid = posting[0]
                term_pids.add(pid)
        
        # docs = intersection of term_pids
        if i == 0:
            docs = term_pids
        else:
            docs = docs.intersection(term_pids)
            
    docs = list(docs)
    return docs

To test whether `search()` is performing the intersection between sets correctly, we use a small test:

In [6]:
matches_cotton = search("cotton", index)
matches_pants = search("pants", index)
matches_both = search("cotton and pants", index)

set(matches_cotton) & set(matches_pants) == set(matches_both)

True

In [7]:
# Search for test queries using "search" function
test_queries = [
    "men's cotton checkered shirt",
    "women casual shirt linen",
    "black sports shoes",
    "soft jacket for women",
    "grey hoodie with pockets",
]

for query in test_queries:
    docs = search(query, index)
    ratio = len(docs) / len(index_info)
    
    print(
        f"{query!r} : {len(docs)} products, "
        f"match rate = {ratio:.2%}"
    )

"men's cotton checkered shirt" : 401 products, match rate = 1.43%
'women casual shirt linen' : 70 products, match rate = 0.25%
'black sports shoes' : 94 products, match rate = 0.33%
'soft jacket for women' : 52 products, match rate = 0.19%
'grey hoodie with pockets' : 28 products, match rate = 0.10%


## <a id='toc3_3_'></a>[3. TF-IDF algorithm](#toc0_)

Our index is a nested dictionary. On the first level, the keys are the terms (e.g. `cotton`), and on the second the keys are the fields where this term shows up (e.g. `title`). The value for the fields is a list of postings `[pid, position]` specifying in which product and position the term appears.

```
index["cotton"] = {
    "title": [
        [pid1, positions],
        [pid2, positions],
    ],
    "description": [
        [doc1, positions],
    ],
    ...
}
```

In [8]:
def create_index_tfidf(index_df: pd.DataFrame, num_documents: int):  
    """
    Implement the inverted index and compute tf, df and idf
    
    :param index_df: DataFrame storing pre-processed product information 
    :returns index: the inverted index (implemented through a nested Python dictionary) containing terms and fields as keys and the corresponding
    list of documents where these keys appears in (and the positions) as values.
    :returns tf: normalized term frequency for each term in each document
    :returns df: number of documents each term appear in
    :returns idf: inverse document frequency of each term
    """
    index = defaultdict(dict)
    # tf is also a dictionary of dictionaries e.g. tf["cotton"]["title"][pid] = TF value
    tf = defaultdict(lambda: defaultdict(dict)) 
    # tf = defaultdict(list)  # term frequencies of terms in documents (documents in the same order as in the main index)
    df = defaultdict(int)  # document frequencies of terms in the corpus
    idf = defaultdict(float)

    # Iterate over each row (products)
    for i, row in index_df.iterrows():
        pid = row['pid'] # Product's ID

        # Build index only for this product and field
        # E.g. ['solid', 'women'] --> 'solid': {'title': [pid, [0]]}, 'women': {'title': [pid, [1]]}
        current_product_index = {}

        # Iterate over fields in this row: title, description, brand, ...
        for field in fields:

            terms = row[field] # List of preprocessed terms

            for position, term in enumerate(terms):
                if term in current_product_index.keys():

                    if field in current_product_index[term].keys():
                        # if the term is already in the current_product_index for the current page (current_page_current_product_index)
                        # append the position to the corresponding list
                        current_product_index[term][field][1].append(position)
                    else:
                        current_product_index[term][field] = [pid, array('I',[position])]              
                
                else:
                    # term is not in the dictionary yet
                    current_product_index[term] = {field: [pid, array('I',[position])]} #'I' indicates unsigned int (int in Python)}
                        
        # Normalize term frequencies
        # Compute the denominator to normalize term frequencies.
        # norm is the same for all terms of a document.
        norm = 0
        for term in current_product_index.keys():
            # sum over all the fields for the current term to get the total frequency of the term in the current document
            norm += sum(
                len(posting[1])
                for posting in current_product_index[term].values()
            ) ** 2
        norm = math.sqrt(norm)

        for term in current_product_index.keys():
            for field, posting in current_product_index[term].items():
                # append the tf for current term (tf = term frequency in current doc/norm)
                tf[term][field][pid] = np.round(len(posting[1])/norm, 4) 
        
            df[term] += 1 # increment DF for current term
            
        # Merge the current product index with the main index
        for term, field_dict in current_product_index.items():
            for field, posting_page in field_dict.items():
                index[term].setdefault(field, []).append(posting_page)

    # Compute IDF using the df we calculated before
    for term in df:
        idf[term] = np.round(np.log(float(num_documents/df[term])), 4)

    return index, tf, df, idf

index, tf, df, idf = create_index_tfidf(index_info, num_documents=len(index_info))

In the previous lab, we mentioned we kept the fields separate to have the possibility of giving different weights to each. For now, we will give equal weight to all of them, but we could tune them in the future if needed.

In [9]:
# For now, we give equal weights to all fields (1/7)
equal_weights = np.array(len(fields)*[1/len(fields)])

for f, w in zip(fields, equal_weights):
    print(f"{f:<20} = {w:>10.4f}")

title                =     0.1429
description          =     0.1429
brand                =     0.1429
category             =     0.1429
sub_category         =     0.1429
seller               =     0.1429
product_details      =     0.1429


In [10]:
field_weights = dict(zip(fields, equal_weights))

In [ ]:
# TODO: finish this part
# implement field weights (if all weights are equal the result should be the same as regular TF-IDF)

def rank_documents(terms, docs, index, idf, tf, fields=fields, field_weights=equal_weights):
    """
    Perform the ranking of the results of a search based on the tf-idf weights

    :param terms: list of query terms
    :param docs: list of documents, to rank, matching the query
    :param index: inverted index data structure
    :param idf: inverted document frequencies
    :param tf: term frequencies
    :param fields: list of fields where terms can be found (e.g ['title', 'description', ...])
    :param field_weights: 

    Returns:
    Print the list of ranked documents
    """
    
    # Empty query or no doc case
    if not terms or not docs:
        return []
    
    # Create dict with fields weights
    if len(field_weights) != len(fields):
        raise ValueError(f"Number of fields weights ({len(field_weights) }) must be the same as number of fields ({len(fields)})")
    weights = dict(zip(fields, field_weights))
    
    # I'm interested only on the element of the docVector corresponding to the query terms
    # The remaining elements would became 0 when multiplied to the query_vector
    doc_vectors = defaultdict(lambda: [0] * len(terms)) # I call doc_vectors[k] for a nonexistent key k, the key-value pair (k,[0]*len(terms)) will be automatically added to the dictionary
    query_vector = [0] * len(terms)

    # compute the norm for the query tf
    query_terms_count = Counter(terms)  # get the frequency of each term in the query.
    # Example: collections.Counter(["hello","hello","world"]) --> Counter({'hello': 2, 'world': 1})

    query_norm = la.norm(list(query_terms_count.values()))

    # Avoid duplicates in the query vector
    for termIndex, term in enumerate(query_terms_count):  #termIndex is the index of the term in the query
        if term not in index:
            continue

        ## Compute tf*idf(normalize TF as done with documents)
        query_vector[termIndex]=query_terms_count[term]/query_norm * idf[term] # query_vector[0] corresponds to the first term in the query

        # Generate doc_vectors for matching docs
        # index[term][field] contains [doc, positions] records.
        for field, postings in index[term].items():
            for doc, positions in postings:
                if doc in docs:

                    # Sum TF-IDF contributions from all fields
                    doc_vectors[doc][termIndex] += (
                        tf[term][field][doc] * idf[term] * weights[field]
                    )
        
    # Calculate the score of each doc

    # compute the cosine similarity between queyVector and each docVector:
    # HINT: you can use the dot product because in case of normalized vectors it corresponds to the cosine similarity
    # see np.dot
    # TODO: question, shouldn't we normalize here?
    doc_scores=[[np.dot(curDocVec, query_vector), doc] for doc, curDocVec in doc_vectors.items() ]
    doc_scores.sort(reverse=True)
    result_docs = [x[1] for x in doc_scores]
    return result_docs

In [12]:
def search_tf_idf(query, index, fields=fields, field_weights=equal_weights):
    """
    output is the list of documents that contain any of the query terms.
    So, we will get the list of documents for each query term, and take the union of them.
    """
    docs = search(query, index)
    query = build_terms(query)

    ranked_docs = rank_documents(query, docs, index, idf, tf, fields, field_weights)
    return ranked_docs

In [13]:
def display_results(query, ranked_docs, df, top=10):
    print(f"🔎 You searched for '{query}'")
    print(f"\n====================== Top {top} results out of {len(ranked_docs)} ====================== \n")
    
    df_idx = df.set_index("pid")
    
    for rank, d_id in enumerate(ranked_docs[:top], 1):
        if d_id not in df_idx.index:
            continue
            
        row = df_idx.loc[d_id]
        
        # Extract and clean data
        title = row['title']
        brand = str(row.get('brand', 'N/A')) if pd.notna(row.get('brand')) else 'N/A'
        category = str(row.get('category', 'N/A')) if pd.notna(row.get('category')) else 'N/A'
        sub_cat = str(row.get('sub_category', '')) if pd.notna(row.get('sub_category')) else ''
        
        raw_desc = str(row['description']).replace('\n', ' ').replace('\r', ' ') if pd.notna(row['description']) else ""
        desc = raw_desc[:70] + "..." if len(raw_desc) > 70 else raw_desc
        
        # 1. Rank, Title, Brand, Cat > Subcat, PID
        cat_string = f"{category} > {sub_cat}" if sub_cat else category
        print(f"\033[1m\033[96m{rank} {title}\033[0m | \033[95m{brand}\033[0m | \033[94m{cat_string}\033[0m\033[2m | {d_id}\033[0m")

        # 2. Description in grey
        print(f"   \033[90m{desc}\033[0m\n")    
        

In [ ]:
# Try one test query
products = pd.read_csv("../../data/fashion_products.csv")
query = test_queries[2]
ranked_docs = search_tf_idf(query, index)
display_results(query, ranked_docs, products, top=7)

🔎 You searched for 'black sports shoes'

====================== Top 7 results out of 94 ====================== 

1 JAGUAR Football Shoes For Women  (Red, Black) | Vector | Footwear > Men's Footwear | SHOFN3EJZKBG6QUF
   sports shoes

2 Lace Up For Men  (Black) | Bone | Footwear > Men's Footwear | SHOFVXG5XR59AFTH
   Bonexy, a brand well known for its domination over the footwear indust...

3 Lace Up For Men  (Black) | Bone | Footwear > Men's Footwear | SHOFVJX2TP8FRR7P
   Bonexy, a brand well known for its domination over the footwear indust...

4 Loafers For Women  (Black) | Bone | Footwear > Men's Footwear | SHOFX2G6FHVNAHZN
   Bonexy, a brand well known for its domination over the footwear indust...

5 Loafers For Men  (Black) | Bone | Footwear > Men's Footwear | SHOFVHY8MAYUPTZF
   Bonexy, a brand well known for its domination over the footwear indust...

6 RS-7250 Running Shoes For Men  (Black) | Vector | Footwear > Men's Footwear | SHOFN3EJHXHGSGUC
   sports shoes

7 Sneakers For

In [15]:
# Give terms in the title twice the weight of terms in other fields
field_weights = [2, 1, 1, 1, 1, 1, 1]
field_weights = field_weights / la.norm(field_weights)

# Try the same query again
query = test_queries[2]
ranked_docs = search_tf_idf(query, index, field_weights=field_weights)
display_results(query, ranked_docs, products, top=7)

🔎 You searched for 'black sports shoes'

====================== Top 7 results out of 94 ====================== 

1 JAGUAR Football Shoes For Women  (Red, Black) | Vector | Footwear > Men's Footwear | SHOFN3EJZKBG6QUF
   sports shoes

2 RS-7250 Running Shoes For Men  (Black) | Vector | Footwear > Men's Footwear | SHOFN3EJHXHGSGUC
   sports shoes

3 Cricket Shoes For Women  (White, Black) | Vector | Footwear > Men's Footwear | SHOEWQKVYWWZ2CZK
   

4 Cricket Shoes For Women  (White, Black) | Vector | Footwear > Men's Footwear | SHOEWQKV5PHERHXN
   

5 Football Shoes For Women  (Green, Black) | Vector | Footwear > Men's Footwear | SHOEP5GVHY8DVGZQ
   

6 Breeze Football Shoes For Women  (Green, Black) | Vector | Footwear > Men's Footwear | SHOFFTZNUFSFGGMR
   

7 Football Shoes For Men  (Green, Black) | Vector | Footwear > Men's Footwear | SHOEWQKVXCQRCSSK
   



# <a id='toc4_'></a>[Part 2](#toc0_)

In this section we assess the effectiveness of our retrieval solutions using the following metrics:
1. Precision@K (P@K) 
2. Recall@K (R@K) 
3. Average Precision@K (P@K) 
4. F1-Score@K 
5. Mean Average Precision (MAP) 
6. Mean Reciprocal Rank (MRR) 
7. Normalized Discounted Cumulative Gain (NDCG)

## <a id='toc4_1_'></a>[1. Implementation of evaluation metrics](#toc0_)

In [16]:
# i. Precision@K (P@K)
def precision_at_k(doc_score, y_score, k=10):
    """
    Parameters
    ----------
    doc_score: Ground truth (true relevance labels).
    y_score: Predicted scores.
    k : number of doc to consider.

    Returns
    -------
    precision @k : float

    """
    order = np.argsort(y_score)[::-1]  # Bonus point: How to improve the effitiency of this part?
    doc_score = doc_score[order[:k]]
    relevant = sum(doc_score == 1)
    return float(relevant) / k

In [17]:
# ii. Recall@K (R@K)
def recall_at_k(doc_score, y_score, k=10, number_of_relevant=None):
    """Use the total known relevant count when doc_score contains only retrieved products."""
    doc_score = np.asarray(doc_score)
    if number_of_relevant is None:
        number_of_relevant = np.sum(doc_score == 1)
    if number_of_relevant == 0:
        return 0.0

    order = np.argsort(y_score)[::-1]
    doc_score = doc_score[order[:k]]
    relevant = np.sum(doc_score == 1)
    return float(relevant) / number_of_relevant


In [18]:
# iii. Average Precision@K (P@K)
def avg_precision_at_k(doc_score, y_score, k=10):
    """
    Parameters
    ----------
    
    doc_score: Ground truth (true relevance labels).
    y_score: Predicted scores.
    k : number of doc to consider.

    Returns
    -------
    average precision @k : float
    """
    order = np.argsort(y_score)[::-1]  # get the list of indexes of the predicted score sorted in descending order.

    prec_at_i = 0
    prec_at_i_list = []
    number_of_relevant = 0
    number_to_iterate = min(k, len(order))

    for i in range(number_to_iterate):
        if doc_score[order[i]] == 1:
            number_of_relevant += 1
            prec_at_i = number_of_relevant / (i + 1)
            prec_at_i_list.append(prec_at_i)

    if number_of_relevant == 0:
        return 0
    else:
      return np.sum(prec_at_i_list) / number_of_relevant

In [19]:
# iv. F1-Score@K
def f1_at_k(doc_score, y_score, k=10, number_of_relevant=None):
    """Harmonic mean of precision@k and recall@k."""
    doc_score = np.asarray(doc_score)
    precision = precision_at_k(doc_score, y_score, k)
    recall = recall_at_k(doc_score, y_score, k, number_of_relevant)

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


In [20]:
# v. Mean Average Precision (MAP)
def map_at_k(search_res, k=10):
    """
    Parameters
    ----------
    search_res: search results dataset containing:
        query_id: query id.
        doc_id: document id.
        predicted_relevance: relevance predicted through LightGBM.
        doc_score: actual score of the document for the query (ground truth).

    Returns
    -------
    mean average precision @ k : float
    """
    avp = []
    for q in search_res["query_id"].unique():  # loop over all query id
        curr_data = search_res[search_res["query_id"] == q]  # select data for current query
        avp.append(avg_precision_at_k(np.array(curr_data["is_relevant"]),
                   np.array(curr_data["predicted_relevance"]), k))  #append average precision for current query
    return np.sum(avp) / len(avp), avp  # return mean average precision

In [21]:
# vi. Mean Reciprocal Rank (MRR)
def rr_at_k(doc_score, y_score, k=10):
    """
    Parameters
    ----------
    doc_score: Ground truth (true relevance labels).
    y_score: Predicted scores.
    k : number of doc to consider.

    Returns
    -------
    Reciprocal Rank for qurrent query
    """

    order = np.argsort(y_score)[::-1]  # get the list of indexes of the predicted score sorted in descending order.
    doc_score = np.take(doc_score, order[:k])  # sort the actual relevance label of the documents based on predicted score(hint: np.take) and take first k.
    if np.sum(doc_score) == 0:  # if there are not relevant doument return 0
        return 0
    return 1.0 / (np.argmax(doc_score == 1) + 1)  # hint: to get the position of the first relevant document use "np.argmax"


def mrr_at_k(search_res, k=10):
    """Mean RR@k across queries; columns match map_at_k."""
    rr = []
    for q in search_res["query_id"].unique():
        curr_data = search_res[search_res["query_id"] == q]
        rr.append(rr_at_k(
            np.array(curr_data["is_relevant"]),
            np.array(curr_data["predicted_relevance"]), k
        ))

    if len(rr) == 0:
        return 0.0, rr
    return np.mean(rr), rr


In [22]:
# vii. Normalized Discounted Cumulative Gain (NDCG)
def dcg_at_k(doc_score, y_score, k=10):
    order = np.argsort(y_score)[::-1]  # get the list of indexes of the predicted score sorted in descending order.
    doc_score = np.take(doc_score, order[:k])  # sort the actual relevance label of the documents based on predicted score(hint: np.take) and take first k.
    gain = 2 ** doc_score - 1  # Compute gain (use formula 7 above)
    discounts = np.log2(np.arange(len(doc_score)) + 2)  # Compute denominator
    return np.sum(gain / discounts)  #return dcg@k


def ndcg_at_k(doc_score, y_score, k=10):
    dcg_max = dcg_at_k(doc_score, doc_score, k)
    if not dcg_max:
        return 0
    return np.round(dcg_at_k(doc_score, y_score, k) / dcg_max, 4)

## <a id='toc4_2_'></a>[2. Applying the evaluation metrics](#toc0_)

First we read `validation_labels.csv` and see how many labels we have for each query.

In [23]:
validation_labels = pd.read_csv("../../data/validation_labels.csv")
display(validation_labels.head())

queries = {
    1: "women full sleeve sweatshirt cotton",
    2: "men slim jeans blue",
}

print(f"The annotated dataset has {len(validation_labels)} ratings in total.")
validation_labels.groupby('query_id').count()['labels']

,title,pid,query_id,labels
0,Full Sleeve Printed Women Sweatshirt,SWSFFVKBCQG5FHPF,1,1
1,Full Sleeve Striped Women Sweatshirt,SWSFJY5ZFHQ7HXKW,1,0
2,Full Sleeve Printed Women Sweatshirt,SWSFUY89NHMZHZPX,1,1
3,Full Sleeve Graphic Print Women Sweatshirt,SWSFXQ5YX6RZKHP4,1,1
4,Full Sleeve Solid Women Sweatshirt,JCKFTZBC3DMCVYXH,1,0


The annotated dataset has 40 ratings in total.


query_id
1    20
2    20
Name: labels, dtype: int64

We assume the items in `validation_labels.csv` are already ranked for each query: i.e row 0 is the most relevant product for query 1, row 1 is the second most relevant product, and so on.

In [24]:
def evaluate_queries(queries, labels_df, k=10):
    """
    Computes all evaluation metrics for the queries provided.

    :param queries: dict {query_id: query text}
    :param labels_df: DataFrame with columns 'query_id', 'pid' and 'labels' (1 = relevant, 0 = nonrelevant)
    :param k: cutoff for the @k metrics
    :returns per_query: DataFrame, one row per query, with P@k, R@k, AP@k, F1@k, RR@k and NDCG@k
    :returns summary: one-row DataFrame with MAP@k and MRR@k (means over all the queries)
    """
    rows = {}              # {query_id: {metric name: value}}

    for query_id, query in queries.items():

        # Select the labeled products for the current query
        query_data = labels_df[labels_df["query_id"] == query_id]

        # True relevance labels (1 = relevant, 0 = non-relevant)
        doc_score = query_data.reset_index(drop=True)['labels'].to_numpy()

        # Predicted relevance labels.
        # We already have the final order, so we give decreasing scores (N, N-1, ..., 1) to keep it unchanged.
        y_score = np.arange(len(doc_score), 0, -1, dtype=float)
        
        # Metrics for this query
        rows[query_id] = {
            f"P@{k}": precision_at_k(doc_score, y_score, k),
            f"R@{k}": recall_at_k(doc_score, y_score, k),
            f"AP@{k}": avg_precision_at_k(doc_score, y_score, k),
            f"F1@{k}": f1_at_k(doc_score, y_score, k),
            f"RR@{k}": rr_at_k(doc_score, y_score, k),
            f"NDCG@{k}": ndcg_at_k(doc_score, y_score, k),
        }

    # DataFrame with results per query
    per_query = pd.DataFrame.from_dict(rows, orient="index").astype(float)
    per_query.index.name = "query_id"

    # MAP = mean of the AP of each query, MRR = mean of the RR of each query
    map_score = per_query[f"AP@{k}"].mean()
    mrr_score = per_query[f"RR@{k}"].mean()
    summary = pd.DataFrame({f"MAP@{k}": [map_score], f"MRR@{k}": [mrr_score]}, index=["all queries"])

    return per_query, summary

In [25]:
per_query, summary = evaluate_queries(queries, labels_df=validation_labels, k=10)

# Display metrics using only 3 decimal places
with pd.option_context("display.float_format", "{:.3f}".format):

    print("Evaluation metrics for queries")
    for k, v in queries.items():
        print(f"{k}: '{v}'")
    display(per_query)

    print("Aggregate evaluation metrics")
    display(summary)

Evaluation metrics for queries
1: 'women full sleeve sweatshirt cotton'
2: 'men slim jeans blue'


,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10
query_id,,,,,,
1,0.700,0.538,0.725,0.609,1.000,0.703
2,0.600,0.600,0.680,0.600,1.000,0.615


Aggregate evaluation metrics


,MAP@10,MRR@10
all queries,0.703,1.000


## <a id='toc4_3_'></a>[3. Ground truth for our test queries](#toc0_)

Now we label by hand the results of our 5 test queries and evaluate the system again with those labels.

### <a id='toc4_3_1_'></a>[a. Labelling results manually](#toc0_)

We export a CSV with the top results of each test query, ranked with TF-IDF. The each row of the CSV corresponds to a $(query, product)$ pair, and has the following columns:
- `title` of the product
- `description` of the product. We include it because it's inaccurate to judge the relevance of a product based only on its title.
- `pid` of the product
- `query_id` of the query that was used to search this product
- `labels` is initially empty, we later fill it manually with `1` (relevant) or `0` (nonrelevant).

As before, the products are already ordered for each query.

In [53]:
def get_product_field(pid: str, field: str):
    return products.set_index('pid').loc[pid][field]

In [ ]:
TEST_QUERIES_CSV = "../../data/test_queries_results.csv"
TOP_N = 20   # number of products exported per query

test_queries_dict = {i: q for i, q in enumerate(test_queries, start=1)} # {1: "men's cotton checkered shirt", 2: ..., ...}.

# Do not overwrite the file if it already exists, otherwise the manual labels would be lost
if os.path.exists(TEST_QUERIES_CSV):
    print(f"{TEST_QUERIES_CSV} already exists, nothing exported. Delete it manually to generate it again.")

# If it doesn't exist, export top N products of each query to csv
else:
    rows = []
    for query_id, query in test_queries_dict.items():
        ranked_docs = search_tf_idf(query, index)[:TOP_N]       # top TOP_N pids for this query, best first

        for pid in ranked_docs:
            rows.append({
                "title": get_product_field(pid, 'title'),       # original title
                "description": get_product_field(pid, 'description'),       # original description
                "pid": pid,
                "query_id": query_id,
                "labels": "",                    # to be filled in manually
            })

    export_df = pd.DataFrame(rows, columns=["title", "description", "pid", "query_id", "labels"])
    export_df.to_csv(TEST_QUERIES_CSV, index=False)

    print(f"Exported {len(export_df)} rows to {TEST_QUERIES_CSV}")
    print(export_df.groupby("query_id").size().rename("rows per query").to_string())

Exported 100 rows to ../../data/test_queries_results.csv
query_id
1    20
2    20
3    20
4    20
5    20


### <a id='toc4_3_2_'></a>[b. Evaluation metrics for our test queries](#toc0_)

In [52]:
k = 10

# Read the CSV after filling in the labels
test_labels = pd.read_csv(TEST_QUERIES_CSV)

# Every product must have a label, and it must be 0 or 1 (empty cells are read as NaN)
invalid = ~test_labels["labels"].isin([0, 1])
if invalid.any():
    raise ValueError(f"{invalid.sum()} rows in {TEST_QUERIES_CSV} have a missing or non-binary label (expected 0 or 1)")

# evaluate_queries expects the columns 'query_id', 'pid' and 'labels'
test_labels = test_labels.rename(columns={"test_query_id": "query_id"})
test_labels["labels"] = test_labels["labels"].astype(int)

per_query_test, summary_test = evaluate_queries(test_queries_dict, test_labels, k)


# Display metrics using only 3 decimal places
with pd.option_context("display.float_format", "{:.3f}".format):

    print("Evaluation metrics for queries")
    for k, v in test_queries_dict.items():
        print(f"{k}: '{v}'")
    display(per_query_test)

    print("Aggregate evaluation metrics")
    display(summary_test)

Evaluation metrics for queries
1: 'men's cotton checkered shirt'
2: 'women casual shirt linen'
3: 'black sports shoes'
4: 'soft jacket for women'
5: 'grey hoodie with pockets'


,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10
query_id,,,,,,
1,1.000,0.500,1.000,0.667,1.000,1.000
2,1.000,0.500,1.000,0.667,1.000,1.000
3,0.500,0.333,0.563,0.400,1.000,0.507
4,0.300,1.000,0.216,0.462,0.125,0.425
5,0.800,0.571,0.932,0.667,1.000,0.851


Aggregate evaluation metrics


,MAP@10,MRR@10
all queries,0.742,0.825


<!-- comment on each of the evaluation metrics, stating how they differ, and which 
information gives each of them. Analyze your results -->

**Precision@10** is perfect for the first two queries: the first 10 products are all relevant. q3 bad -> we get "lace up shoes" even though we asked for sports shoes (bc. sports is in the description). something similar for q4, and 5 we get some irrelevant products because query words are in the description push these up in the ranking.

**Recall@10** is relatively low for 1 and 2, but thats just because all 20 products we exported are relevant, so in the top 10 we're only getting half of them. So it's not necessarily bad in this case. recall is perfect in q4 because we have only 3 relevant products and they're included in the top 10.

**AP@10** similar pattern to precision.

**F1@10** don't know what to say here.

**RR@10** is 1 for q1, q2, q3, and q5 because the highest rnked product is relevant to the search. in q4, the first relevant product is found in position 8, leading to an RR of $1\over8$ 

**NDGC@10** is perfect for q1 and q2 because of what we mentoned before. In q5 it is relatively good, it's note berfect because some irrelevant products are placed higher than other relevant products in the top 10.


Moving on to aggregate metrics.

**MAP@10** don't know what to say here

**MRR@10** don't know what to say here


Note that this is just a small test with few queries and . we limited it to 20 retrieved items per query because we didn't have the time to annotte more products manually. ideally this hsould be done with multiple annotators, more queries and all retrieved products.

### <a id='toc4_3_3_'></a>[c. Analysis of the current system](#toc0_)

#TODO: Analyze the current search system and identify its main problems or limitations. 
For each issue you find, propose possible ways to resolve it. Consider aspects 
such as retrieval accuracy, ranking quality, handling of different field types, query 
formulation, and indexing strategies. 

potential solution to increase precision --> increase weight of titles. we've seen how some words appear in description that move irrelevant items to higher positions.

so we could do gridsearch 
